# routes.init

> Top-level router assembly for the session management library.

`init_session_manager_routers()` is the single public entry point that hosts call. It:

1. Instantiates mutable state (item list, VC config/state/ids, URL bundle)
2. Creates the virtual collection router (via `init_virtual_collection_router`)
3. Creates the session router (via `init_session_router`)
4. Wires up the `refresh_items` / `refresh_items_oob` closures that hold the state together
5. Populates the `SessionManagementUrls` bundle from the registered routes
6. Returns a `SessionManagementResult` dataclass with everything the host needs

The pattern mirrors `init_management_routers()` from `cjm-transcript-workflow-management` but is simpler — no selection state, no sub-routers for export/import, no forward-ref patching of the cell renderer (the renderer is built once with the populated URLs).

In [ ]:
#| default_exp routes.init

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, Dict, List, Optional, Tuple

from fasthtml.common import Script
from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.routes.router import init_virtual_collection_router
from cjm_fasthtml_virtual_collection.routes.handlers import build_items_changed_response
from cjm_fasthtml_virtual_collection.js.auto_fit import auto_fit_callback_name

from cjm_fasthtml_workflow_session_management.models import (
    SessionManagementUrls, SessionManagementResult, ColumnSpec,
)
from cjm_fasthtml_workflow_session_management.html_ids import SessionManagerHtmlIds
from cjm_fasthtml_workflow_session_management.services.management import SessionManagementService
from cjm_fasthtml_workflow_session_management.components.session_list import (
    build_session_columns, create_session_cell_renderer, render_session_list,
)
from cjm_fasthtml_workflow_session_management.components.page_renderer import (
    render_session_manager_page,
)
from cjm_fasthtml_workflow_session_management.routes.sessions import init_session_router
from cjm_fasthtml_interactions.core.state_store import set_session_id

In [ ]:
#| export
def init_session_manager_routers(
    service:SessionManagementService, # Service for session CRUD
    workflow_url:str, # URL to redirect to on resume (e.g., the workflow root)
    prefix:str="/manage/sessions", # Base prefix for all session management routes
    column_specs:Optional[List[ColumnSpec]]=None, # Host enricher column specs
    get_step_title:Optional[Callable[[str], str]]=None, # Optional step ID -> human title mapper
    page_title:str="Sessions", # Page header title
    page_icon:str="layers", # Lucide icon for the page header
    tab_entries:Optional[List[Tuple[str, str, str, str]]]=None, # Cross-management tab entries
) -> SessionManagementResult: # Assembled result with routers, urls, and render callables
    """Initialize all session management routers with virtual collection integration."""
    # --- URL bundle (populated after route registration) ---
    urls = SessionManagementUrls(
        management_page="", list_sessions="", session_detail="",
        create_session="", delete_session="", rename_session="", resume_session="",
    )
    
    # --- Shared mutable state ---
    _items:List[Any] = []
    # Holds the current active session ID so the cell renderer can draw the "Active" badge
    # without needing the request. Updated in _refresh_items from the latest HTTP session.
    _active_ref = {"id": ""}
    
    def _get_active_id() -> str:
        return _active_ref["id"]
    
    # --- VC configuration ---
    vc_config = VirtualCollectionConfig(
        prefix=SessionManagerHtmlIds.VC_PREFIX,
        columns=build_session_columns(column_specs),
    )
    vc_state = VirtualCollectionState(visible_rows=1, cursor_index=-1)
    vc_ids = VirtualCollectionHtmlIds(prefix=vc_config.prefix)
    vc_btn_ids = VirtualCollectionButtonIds(prefix=vc_config.prefix)
    _refit = auto_fit_callback_name(vc_config)
    
    # --- Cell renderer (built once; URLs are populated after route registration but
    # the renderer holds `urls` by reference, so it picks up the final values at call time). ---
    render_cell = create_session_cell_renderer(
        urls=urls,
        get_active_session_id=_get_active_id,
        get_step_title=get_step_title,
    )
    
    # --- Core callbacks ---
    def _refresh_items(request=None):
        """Reload sessions from the service and update the active session pointer."""
        _items[:] = service.list_sessions()
        vc_state.total_items = len(_items)
        vc_state.window_start = 0
        vc_state.cursor_index = 0 if _items else -1
        if request is not None:
            try:
                _active_ref["id"] = get_session_id(request.session)
            except Exception:
                _active_ref["id"] = ""
    
    def _refresh_items_oob(request=None):
        """Reload sessions and return OOB tuple for HTMX."""
        _refresh_items(request=request)
        return build_items_changed_response(
            _items, vc_state, vc_config, vc_ids, render_cell,
            focus_url=vc_urls.focus_row, refit_callback=_refit,
        )
    
    # Request-less variant for contexts where we don't have one (tests, stubs).
    def _refresh_items_noreq():
        _refresh_items(request=None)
    
    def _refresh_items_oob_noreq():
        return _refresh_items_oob(request=None)
    
    def _sort_callback(items_list, column_key, ascending):
        """Re-fetch sorted items from the service rather than sorting in place."""
        if column_key == "label":
            order_by = "label"
        elif column_key == "status":
            # Sorting by step maps to sorting by updated_at — close enough for ordering
            # sessions within the same step grouping naturally.
            order_by = "updated_at"
        elif column_key == "updated":
            order_by = "updated_at"
        else:
            order_by = "updated_at"
        items_list[:] = service.list_sessions(order_by=order_by, descending=not ascending)
    
    # --- VC router ---
    vc_router, vc_urls = init_virtual_collection_router(
        config=vc_config,
        state_getter=lambda: vc_state,
        state_setter=lambda s: None,
        get_items=lambda: _items,
        render_cell=render_cell,
        sort_callback=_sort_callback,
        route_prefix=f"{prefix}/vc",
    )
    
    # --- Render closures ---
    def _render_list():
        return render_session_list(
            items=_items,
            vc_config=vc_config,
            vc_state=vc_state,
            vc_ids=vc_ids,
            vc_btn_ids=vc_btn_ids,
            vc_urls=vc_urls,
            mgmt_urls=urls,
            render_cell=render_cell,
        )
    
    def _render_page():
        return render_session_manager_page(
            urls=urls,
            render_list_fn=_render_list,
            title=page_title,
            icon_name=page_icon,
            tab_entries=tab_entries,
            active_tab="sessions",
        )
    
    # --- Session router ---
    sess_router, sess_routes = init_session_router(
        service=service,
        prefix=prefix,
        urls=urls,
        workflow_url=workflow_url,
        refresh_items=_refresh_items_noreq,
        refresh_items_oob=_refresh_items_oob_noreq,
        render_page=_render_page,
        render_list=_render_list,
    )
    
    # --- Populate URL bundle from registered routes ---
    urls.management_page = sess_routes["management_page"].to()
    urls.list_sessions = sess_routes["list_sessions"].to()
    urls.create_session = sess_routes["create_session"].to()
    urls.delete_session = sess_routes["delete_session"].to()
    urls.rename_session = sess_routes["rename_session"].to()
    urls.resume_session = sess_routes["resume_session"].to()
    # session_detail is reserved for a future detail view.
    
    # --- Host-facing resume helper ---
    def _resume_session(sess:Any, session_id:str) -> None:
        """Set the active workflow session ID in the host HTTP session."""
        set_session_id(sess, session_id)
    
    return SessionManagementResult(
        routers=[sess_router, vc_router],
        urls=urls,
        routes=sess_routes,
        render_page=_render_page,
        render_list=_render_list,
        refresh_items=_refresh_items_noreq,
        resume_session=_resume_session,
    )

In [ ]:
# Integration smoke test: assemble a full router bundle and exercise the render closures.
# Note: FastHTML registers routers via `fast_app(router, *routers, ...)` at construction
# time, not via include_router afterward. The .to() URLs on APIRouter routes resolve from
# the APIRouter's own prefix, so they're populated by init_session_manager_routers()
# without needing app registration here.
import tempfile, os
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

_tmp = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
_tmp_path = _tmp.name
_tmp.close()
_store = SQLiteWorkflowStateStore(_tmp_path)
_service = SessionManagementService(
    _store, "integration_test",
    enricher=lambda s: {"sources": "1"},
)

# Seed a session so the list isn't empty.
_sid = _service.create_session(label="Test Session")

result = init_session_manager_routers(
    service=_service,
    workflow_url="/workflow/",
    prefix="/manage/sessions",
    column_specs=[ColumnSpec(field="sources", header="Sources")],
    page_title="Test Sessions",
    tab_entries=[
        ("documents", "Documents", "file-text", "/manage/documents"),
        ("sessions", "Sessions", "layers", "/manage/sessions"),
    ],
)

# Route handler names.
expected = {"management_page", "list_sessions", "create_session",
            "delete_session", "rename_session", "resume_session"}
assert set(result.routes.keys()) == expected, f"routes mismatch: {set(result.routes.keys())}"

# URL bundle populated (URLs come from APIRouter.to() which resolves via the router prefix).
assert result.urls.management_page, f"management_page URL not populated: {result.urls.management_page!r}"
assert result.urls.create_session, f"create_session URL not populated: {result.urls.create_session!r}"
assert result.urls.delete_session, f"delete_session URL not populated: {result.urls.delete_session!r}"
assert result.urls.rename_session, f"rename_session URL not populated: {result.urls.rename_session!r}"
assert result.urls.resume_session, f"resume_session URL not populated: {result.urls.resume_session!r}"
print(f"URLs: mgmt={result.urls.management_page}")
print(f"      create={result.urls.create_session}")
print(f"      delete={result.urls.delete_session}")
print(f"      resume={result.urls.resume_session}")

# Refresh and render: should not raise.
result.refresh_items()
page = result.render_page()
assert page is not None
list_frag = result.render_list()
assert list_frag is not None
assert list_frag.attrs.get("id") == SessionManagerHtmlIds.SESSION_LIST

# Service sanity: our seeded session is visible through the service.
sessions = _service.list_sessions()
assert len(sessions) == 1
assert sessions[0].resolved_label == "Test Session"
assert sessions[0].enriched_fields == {"sources": "1"}

# Router count: sessions router + VC router = 2.
assert len(result.routers) == 2

os.unlink(_tmp_path)
print(f"\nIntegration OK: {len(result.routers)} routers, {len(result.routes)} routes")

URLs: mgmt=/manage/sessions/management_page
      create=/manage/sessions/create_session
      delete=/manage/sessions/delete_session
      resume=/manage/sessions/resume_session

Integration OK: 2 routers, 6 routes


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()